In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
EDA_TABLES_DIR = PROCESSED_DIR / "eda_tables"

In [1]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [2]:
import urllib.parse
from sqlalchemy import create_engine, text

server = r"AHMED_KROUSH\SQLEXPRESS"
database = "AirbnbDW"
driver = "ODBC Driver 18 for SQL Server"

params = urllib.parse.quote_plus(
    f"DRIVER={{{driver}}};SERVER={server};DATABASE={database};"
    "Trusted_Connection=yes;TrustServerCertificate=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

with engine.connect() as conn:
    print(conn.execute(text("SELECT DB_NAME()")).scalar())
    print(conn.execute(text("SELECT name FROM sys.tables")).fetchall())

AirbnbDW
[('dim_city',), ('dim_host_segment',), ('dim_price_tier',), ('dim_listing',), ('fact_listing',)]


In [3]:
import pandas as pd
from sqlalchemy import text

enriched = pd.read_csv(EDA_TABLES_DIR / "Airbnb_Enriched.csv", dtype={"listing_id": str})
dim_city = pd.read_csv(EDA_TABLES_DIR / "dim_city.csv")

dim_host_segment = pd.DataFrame({
    "host_segment": ["Superhost + Guest Favorite", "Superhost only", "Guest Favorite only", "Standard"],
    "is_superhost": [1, 1, 0, 0],
    "is_guest_favorite": [1, 0, 1, 0],
})

dim_price_tier = pd.DataFrame({"price_tier": ["Budget", "Mid-range", "Upper", "Luxury"]})

dim_listing = enriched[["listing_id", "Name", "URL"]].rename(columns={"Name": "listing_name", "URL": "url"})
dim_listing["listing_id"] = dim_listing["listing_id"].astype("int64")
dim_listing["url"] = dim_listing["url"].str.split("?").str[0]

dim_city.to_sql("dim_city", engine, if_exists="append", index=False)
dim_host_segment.to_sql("dim_host_segment", engine, if_exists="append", index=False)
dim_price_tier.to_sql("dim_price_tier", engine, if_exists="append", index=False)
dim_listing.to_sql("dim_listing", engine, if_exists="append", index=False)

with engine.connect() as conn:
    for t in ["dim_city", "dim_host_segment", "dim_price_tier", "dim_listing"]:
        print(t, conn.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar())

dim_city 15
dim_host_segment 4
dim_price_tier 4
dim_listing 636


In [4]:
with engine.connect() as conn:
    host_ids = pd.read_sql("SELECT host_segment_id, host_segment FROM dim_host_segment", conn)
    tier_ids = pd.read_sql("SELECT price_tier_id, price_tier FROM dim_price_tier", conn)

fact = (enriched
        .merge(dim_city[["city", "city_id"]], on="city", how="left")
        .merge(host_ids, on="host_segment", how="left")
        .merge(tier_ids, on="price_tier", how="left"))

fact = fact[["listing_id", "city_id", "host_segment_id", "price_tier_id",
             "Price_total_USD", "Price_per_night_USD", "Rating", "Reviews",
             "price_vs_city_median_pct", "has_rating"]].rename(columns={
    "Price_total_USD": "price_total_usd",
    "Price_per_night_USD": "price_per_night_usd",
    "Rating": "rating",
    "Reviews": "reviews",
})
fact["listing_id"] = fact["listing_id"].astype("int64")

print(fact[["city_id", "host_segment_id", "price_tier_id"]].isna().sum())
print(fact.shape)

city_id            0
host_segment_id    0
price_tier_id      0
dtype: int64
(636, 10)


In [5]:
fact.to_sql("fact_listing", engine, if_exists="append", index=False)

with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*) FROM fact_listing")).scalar())
    print(conn.execute(text("SELECT COUNT(*) FROM fact_listing WHERE rating IS NULL")).scalar())

636
80
